# Aprendizado de Máquina — Lista prática 07

## Pré-processamento e *Pipelines*

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta lista tem um cenário deliberadamente cruel: em dois dos exercícios a
resposta $y$ é gerada **sem nenhuma relação com $X$**. Qualquer $R^2$ acima de
zero é ilusão pura, e dá para medir exatamente quanto de ilusão cada erro de
conduta produz.

> **a pergunta que separa vazamento grave de leve não é ``quanto a etapa aprende
> dos dados''. É ``a etapa olha o $Y$?''.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o vazamento grave: selecionar olhando o $Y$

Gere $n=60$ observações com $d=3000$ covariáveis **e uma resposta independente de
todas elas**. Depois selecione as 20 covariáveis mais associadas a $y$ de duas
formas: uma vez antes da validação cruzada, outra vez dentro do `Pipeline`.

O $R^2$ verdadeiro de qualquer modelo aqui é **zero**.

In [ ]:
rng = np.random.default_rng(2026)
n, d, k, REPETICOES = 60, 3000, 20, 20
cv = skm.KFold(5, shuffle=True, random_state=0)

fora, dentro = [], []
for _ in range(REPETICOES):
    X = rng.normal(size=(n, d))
    y = rng.normal(size=n)                                    # (a) NENHUMA relacao com X

    # ERRADO: seleciona olhando TODOS os dados, e so entao valida
    selecao = SelectKBest(f_regression, k=k).fit(X, y)
    fora.append(skm.cross_val_score(skl.LinearRegression(),
                                    selecao.transform(X), y,  # (b)
                                    cv=cv, scoring="r2").mean())

    # CERTO: a selecao e um passo do pipeline, refeita em cada dobra
    tubo = Pipeline([("sel", SelectKBest(f_regression, k=k)),  # (c)
                     ("mqo", skl.LinearRegression())])
    dentro.append(skm.cross_val_score(tubo, X, y, cv=cv, scoring="r2").mean())   # (d)

print(f"selecao FORA da dobra  : R2 = {np.mean(fora):+.4f}")
print(f"selecao DENTRO do tubo : R2 = {np.mean(dentro):+.4f}")

Deve imprimir `selecao FORA da dobra  : R2 = +0.4098` e
`selecao DENTRO do tubo : R2 = -0.6962`.

O primeiro número é a medição central da aula. **Quarenta por cento de $R^2$
fabricados a partir de ruído puro** — e note que este não é um caso artificial
inventado para assustar: $n=60$ e $d=3000$ é a proporção típica de um estudo de
expressão gênica ou de qualquer problema *ômico*.

O segundo número é negativo, e tinha de ser. Fazendo a seleção dentro de cada
dobra, o modelo escolhe covariáveis usando só as 48 observações de treino, e as
12 de validação não têm por que concordar. Um $R^2$ de $-0{,}70$ é o que se
espera de um modelo com 20 parâmetros ajustados a 48 observações de ruído: pior
que chutar a média.

A diferença entre os dois — mais de uma unidade inteira de $R^2$ — é o custo de
mover uma linha de código para dentro do `Pipeline`.

> **Sua vez.** Repita a comparação com $d=100$ em vez de $3000$, mantendo
> $k=20$. O $R^2$ inflado sobe ou desce? Isso confirma ou contraria o
> Exercício 2(b) da Lista Teórica 07?

---
## Exercício 2 — o vazamento leve: padronizar

Agora o mesmo par de condutas, mas com a **padronização** no lugar da seleção — e
num problema com sinal de verdade, para que haja o que estragar. As escalas das
oito covariáveis diferem por quatro ordens de grandeza, que é a situação em que a
padronização mais importa.

In [ ]:
escalas = np.array([1, 50, 0.01, 5, 1, 200, 0.1, 2])
coeficientes = np.r_[1.5, 0.02, 80, 0.3, -1.0, 0.005, 10, 0.4]

fora_esc, dentro_esc = [], []
rng = np.random.default_rng(7)
for _ in range(REPETICOES):
    Xp = rng.normal(size=(n, 8)) * escalas
    yp = Xp @ coeficientes + rng.normal(0, 1, n)

    # ERRADO: o scaler aprende media e desvio no conjunto TODO
    escala = StandardScaler().fit(Xp)                          # (a)
    fora_esc.append(skm.cross_val_score(skl.Ridge(alpha=1.0),
                                        escala.transform(Xp), yp,
                                        cv=cv, scoring="r2").mean())

    # CERTO: o scaler e um passo do pipeline
    tubo = Pipeline([("escala", StandardScaler()),             # (b)
                     ("ridge", skl.Ridge(alpha=1.0))])
    dentro_esc.append(skm.cross_val_score(tubo, Xp, yp, cv=cv, scoring="r2").mean())

print(f"escala FORA da dobra  : R2 = {np.mean(fora_esc):.4f}")
print(f"escala DENTRO do tubo : R2 = {np.mean(dentro_esc):.4f}")
print(f"diferenca             : {abs(np.mean(fora_esc) - np.mean(dentro_esc)):.2e}")   # (c)

Deve imprimir `escala FORA da dobra  : R2 = 0.8594`,
`escala DENTRO do tubo : R2 = 0.8593` e `diferenca: 1.11e-04`.

Os dois números coincidem até a **quarta casa decimal**. Compare com o
Exercício 1, onde a diferença foi de mais de uma unidade de $R^2$: são quatro
ordens de grandeza de distância entre os dois erros.

A razão é a do Exercício 2(c) da Lista Teórica 07 — o `StandardScaler` calcula
duas estatísticas por coluna de $X$ e **nunca vê o $y$**. Sem esse canal, não há
como fabricar acerto. Usar o conjunto todo muda a média e o desvio por algo da
ordem de $1/\sqrt n$, e esse deslocamento afeta todas as observações igualmente.

Isso **não** é licença para padronizar fora do `Pipeline`. É informação sobre
onde gastar atenção: numa revisão de código, a pergunta cara é ``como as
covariáveis foram escolhidas?'', não ``o scaler está no lugar certo?''.

---
## Exercício 3 — o vazamento que nenhum `Pipeline` conserta

Um laboratório mede 40 marcadores em 60 pacientes, **três amostras por
paciente**. A resposta depende só do paciente. Um analista monta o `Pipeline`
correto e roda `KFold` sobre as 180 linhas.

In [ ]:
rng = np.random.default_rng(2026)
n_pacientes, n_amostras, d_marc = 60, 3, 40
grupo = np.repeat(np.arange(n_pacientes), n_amostras)

# cada paciente tem uma "assinatura" propria, medida com ruido em cada amostra
assinatura = rng.normal(0, 1, size=(n_pacientes, d_marc))
Xg = assinatura[grupo] + rng.normal(0, 0.3, size=(n_pacientes * n_amostras, d_marc))

beta = rng.normal(0, 1, d_marc) / np.sqrt(d_marc)
y_por_paciente = assinatura @ beta * 3 + rng.normal(0, 1.0, n_pacientes)
yg = y_por_paciente[grupo] + rng.normal(0, 0.5, n_pacientes * n_amostras)   # (a)

print("linhas:", Xg.shape[0], " pacientes:", n_pacientes)

In [ ]:
modelo = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)

por_linha = skm.cross_val_score(
    modelo, Xg, yg,
    cv=skm.KFold(10, shuffle=True, random_state=0), scoring="r2").mean()

por_grupo = skm.cross_val_score(
    modelo, Xg, yg,
    cv=skm.GroupKFold(10), groups=grupo, scoring="r2").mean()   # (a) e (b)

print(f"KFold (por linha) : R2 = {por_linha:+.4f}")
print(f"GroupKFold        : R2 = {por_grupo:+.4f}")

Para ver o mecanismo, conte quantos pacientes aparecem **dos dois lados** de cada
dobra do `KFold`.

In [ ]:
for i_tr, i_te in skm.KFold(10, shuffle=True, random_state=0).split(Xg):
    nos_dois = len(set(grupo[i_te]) & set(grupo[i_tr]))         # (a)
    print(f"dobra com {len(i_te)} linhas de validacao: {nos_dois} pacientes tambem no treino")

Deve imprimir `KFold (por linha) : R2 = +0.6665` e `GroupKFold : R2 = +0.0341`,
e depois dez linhas mostrando **16 ou 17 pacientes nos dois lados**, de 18 linhas
de validação.

O `KFold` reporta $R^2 = 0{,}67$; o valor honesto é $0{,}03$. O modelo não
aprendeu nada sobre biologia — aprendeu a **reconhecer o paciente**, que ele já
tinha visto no treino, e a devolver o $y$ dele.

A contagem explica por quê: quase toda linha de validação pertence a um paciente
que também está no treino. A chance de as três amostras de um paciente caírem na
mesma dobra de $1/10$ é da ordem de $1\%$, então praticamente nenhum paciente
fica inteiramente de fora.

E o ponto que dá nome ao exercício: **o `Pipeline` está correto e não ajuda em
nada**. Ele controla *quando* cada etapa é ajustada, não *como* as linhas são
repartidas. A ferramenta é o `GroupKFold`, e ela exige que alguém saiba qual
coluna identifica a unidade experimental — informação que não está nos dados, e
sim no desenho do estudo.

---
## Exercício 4 — a receita completa

Feita a parte do que **não** fazer, monte o que se deve fazer: colunas numéricas
com faltantes, colunas categóricas, e um hiperparâmetro a escolher — tudo num
objeto só.

In [ ]:
rng = np.random.default_rng(2026)
n_obs = 800
estado = rng.choice(["RJ", "SP", "MG", "BA"], n_obs, p=[.35, .35, .2, .1])
efeito_estado = {"RJ": 0.0, "SP": 1.5, "MG": -1.0, "BA": 0.5}

idade = rng.normal(40, 12, n_obs)
renda = np.exp(rng.normal(8.5, 0.6, n_obs))
saldo = rng.normal(0, 5000, n_obs)
resposta = (0.05 * idade + 1e-4 * renda + 2e-4 * saldo
            + np.array([efeito_estado[e] for e in estado])
            + rng.normal(0, 1, n_obs))

renda[rng.random(n_obs) < 0.15] = np.nan          # 15% de faltantes

banco = pd.DataFrame({"idade": idade, "renda": renda, "saldo": saldo,
                      "estado": estado, "y": resposta})
print(f"n = {len(banco)}, faltantes em renda: {int(banco['renda'].isna().sum())} "
      f"({100 * banco['renda'].isna().mean():.1f}%)")

In [ ]:
X_banco = banco.drop(columns="y")
y_banco = banco["y"].values

numericas = ["idade", "renda", "saldo"]
categoricas = ["estado"]

preparo = ColumnTransformer([
    ("num", Pipeline([("imputa", SimpleImputer(strategy="median")),      # (a)
                      ("escala", StandardScaler())]), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),        # (b)
])

tubo_final = Pipeline([("preparo", preparo), ("modelo", skl.Ridge())])

busca = skm.GridSearchCV(
    tubo_final,
    {"modelo__alpha": np.logspace(-2, 3, 6)},                            # (c)
    cv=skm.KFold(5, shuffle=True, random_state=2026),
    scoring="r2",
).fit(X_banco, y_banco)

print(f"melhor alpha: {busca.best_params_['modelo__alpha']:g}")
print(f"R2 por CV:    {busca.best_score_:.4f}")
print("colunas geradas:", list(busca.best_estimator_.named_steps["preparo"].get_feature_names_out()))

Deve imprimir `n = 800, faltantes em renda: 118 (14.8%)`, depois
`melhor alpha: 10`, `R2 por CV: 0.6883` e as sete colunas:

```
['num__idade', 'num__renda', 'num__saldo',
 'cat__estado_BA', 'cat__estado_MG', 'cat__estado_RJ', 'cat__estado_SP']
```

Três coisas para reparar.

**As sete colunas.** Três numéricas entram como três; uma categórica de quatro
níveis vira quatro indicadoras. O `get_feature_names_out()` é a forma de saber
qual coeficiente corresponde a quê depois que o `ColumnTransformer` reorganizou
tudo — sem ele, os coeficientes de um pipeline são um vetor sem rótulo.

**A mediana é recalculada 30 vezes**, e não uma: seis valores de `alpha` vezes
cinco dobras (mais um ajuste final no conjunto completo). É o item (b) do
Exercício 3 da Lista Teórica 07.

**`handle_unknown="ignore"`** existe porque `BA` tem só 10% das observações; numa
dobra pode não aparecer nenhuma, e sem esse argumento o `transform` quebraria ao
encontrar a categoria pela primeira vez na validação.

Quanto custaria simplesmente jogar fora a coluna categórica, que é o atalho mais
comum quando não se conhece o `ColumnTransformer`?

In [ ]:
so_numericas = Pipeline([
    ("imputa", SimpleImputer(strategy="median")),
    ("escala", StandardScaler()),
    ("modelo", skl.Ridge(alpha=busca.best_params_["modelo__alpha"])),
])

r2_sem = skm.cross_val_score(so_numericas, X_banco[numericas], y_banco,   # (a)
                             cv=skm.KFold(5, shuffle=True, random_state=2026),
                             scoring="r2").mean()

print(f"R2 com a categorica:  {busca.best_score_:.4f}")
print(f"R2 sem a categorica:  {r2_sem:.4f}")
print(f"perda:                {busca.best_score_ - r2_sem:.4f}")

Deve imprimir `0.6883`, `0.4452` e `perda: 0.2430`.

Jogar fora uma única coluna categórica custa **0,24 de $R^2$** — mais de um terço
do que o modelo conseguia explicar. E era previsível: na geração dos dados, o
efeito de estado varia de $-1{,}0$ (MG) a $+1{,}5$ (SP), uma amplitude de 2,5
contra um ruído de desvio 1.

A moral fecha a aula: o `ColumnTransformer` não é burocracia de organização de
código. Ele é o que torna barato usar colunas que, de outra forma, seriam
descartadas por serem inconvenientes.

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | selecionar variáveis fora da dobra fabrica $R^2 = +0{,}40$ **a partir de ruído puro** |
| 1 | fazendo certo, o $R^2$ é $-0{,}68$ — negativo, como tem de ser |
| 2 | padronizar fora da dobra: $0{,}8594$ contra $0{,}8593$, diferença de $10^{-4}$ |
| 3 | `KFold` sobre linhas do mesmo paciente reporta $0{,}67$ onde o valor honesto é $0{,}03$ |
| 3 | 16 ou 17 dos 18 registros de cada dobra pertencem a um paciente que está no treino |
| 4 | descartar uma coluna categórica custa 0,24 de $R^2$ |

**A seguir.** Fecha o Bloco I. A Aula 08 abre a classificação: a resposta deixa de
ser um número e passa a ser uma classe, o risco deixa de ser erro quadrático, e
aparece um classificador ótimo com nome próprio.